In [7]:
# Import libraries

from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [8]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_CLAIMS_DIR = PROJECT_ROOT / "data" / "01_raw" / "cms_claims"
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "02_preprocessed"
FEATURES_DIR = PROJECT_ROOT / "data" / "03_features"

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_CLAIMS_DIR:", RAW_CLAIMS_DIR)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("FEATURES_DIR:", FEATURES_DIR)

PROJECT_ROOT: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform
RAW_CLAIMS_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\01_raw\cms_claims
PREPROCESSED_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed
FEATURES_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\03_features


In [9]:
# Find all files in the raw CMS claims folder

claim_files = sorted(RAW_CLAIMS_DIR.rglob("*"))

print("Files found:", len(claim_files))

for i, file_path in enumerate(claim_files):
    if file_path.is_file():
        print(i, file_path.relative_to(PROJECT_ROOT))

Files found: 1
0 data\01_raw\cms_claims\.gitkeep


In [10]:
# Find CSV files, including uppercase CSV extensions

csv_files = sorted(
    [
        path for path in RAW_CLAIMS_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() == ".csv"
    ]
)

print("CSV files found:", len(csv_files))

for i, file_path in enumerate(csv_files):
    print(i, file_path.relative_to(PROJECT_ROOT))

CSV files found: 0


In [11]:
# Detect claims files with broader keyword matching

def find_files_by_keywords(files, keywords):
    matches = []

    for file_path in files:
        text = str(file_path).lower()

        if any(keyword.lower() in text for keyword in keywords):
            matches.append(file_path)

    return matches


beneficiary_matches = find_files_by_keywords(
    csv_files,
    [
        "beneficiary",
        "beneficiaries",
        "enrollment",
        "summary",
        "bene",
        "carrier_summary",
        "desynpuf",
    ],
)

inpatient_matches = find_files_by_keywords(
    csv_files,
    [
        "inpatient",
        "inp",
        "ip",
    ],
)

outpatient_matches = find_files_by_keywords(
    csv_files,
    [
        "outpatient",
        "outp",
        "op",
    ],
)

print("Beneficiary matches:", len(beneficiary_matches))
for i, file_path in enumerate(beneficiary_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

print("\nInpatient matches:", len(inpatient_matches))
for i, file_path in enumerate(inpatient_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

print("\nOutpatient matches:", len(outpatient_matches))
for i, file_path in enumerate(outpatient_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

Beneficiary matches: 0

Inpatient matches: 0

Outpatient matches: 0


In [12]:
# Stop early with a clear message if files are missing

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found in {RAW_CLAIMS_DIR}. "
        "Download or unzip the CMS synthetic claims files into this folder first."
    )

if len(inpatient_matches) == 0:
    raise FileNotFoundError(
        "No inpatient claims file found. Check the filenames printed in Cell 5."
    )

if len(outpatient_matches) == 0:
    raise FileNotFoundError(
        "No outpatient claims file found. Check the filenames printed in Cell 5."
    )

# Beneficiary file is useful but not required for this first claims EDA.
beneficiary_file = beneficiary_matches[0] if len(beneficiary_matches) > 0 else None
inpatient_file = inpatient_matches[0]
outpatient_file = outpatient_matches[0]

print("Beneficiary file:", beneficiary_file.relative_to(PROJECT_ROOT) if beneficiary_file else "Not found")
print("Inpatient file:", inpatient_file.relative_to(PROJECT_ROOT))
print("Outpatient file:", outpatient_file.relative_to(PROJECT_ROOT))

FileNotFoundError: No CSV files found in c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\01_raw\cms_claims. Download or unzip the CMS synthetic claims files into this folder first.

In [ ]:
# Load raw claims datasets

beneficiary_df = None

if beneficiary_file is not None:
    beneficiary_df = pd.read_csv(beneficiary_file, low_memory=False)
    print("Beneficiary shape:", beneficiary_df.shape)
    display(beneficiary_df.head())
else:
    print("Beneficiary file not found. Continuing with inpatient and outpatient claims only.")

inpatient_df = pd.read_csv(inpatient_file, low_memory=False)
outpatient_df = pd.read_csv(outpatient_file, low_memory=False)

print("Inpatient shape:", inpatient_df.shape)
print("Outpatient shape:", outpatient_df.shape)

display(inpatient_df.head())
display(outpatient_df.head())

In [ ]:
# Inspect columns for each dataset

def inspect_columns(df, dataset_name):
    columns_df = pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "dtype": [df[col].dtype for col in df.columns],
        "missing_count": [df[col].isna().sum() for col in df.columns],
        "missing_pct": [(df[col].isna().mean() * 100).round(2) for col in df.columns],
    })

    return columns_df.sort_values("missing_pct", ascending=False)


beneficiary_columns = inspect_columns(beneficiary_df, "beneficiary")
inpatient_columns = inspect_columns(inpatient_df, "inpatient")
outpatient_columns = inspect_columns(outpatient_df, "outpatient")

display(beneficiary_columns)
display(inpatient_columns)
display(outpatient_columns)

In [ ]:
# Search columns by keyword

def search_columns(df, keywords):
    matches = []

    for col in df.columns:
        col_lower = col.lower()

        if any(keyword.lower() in col_lower for keyword in keywords):
            matches.append(col)

    return matches


keywords = {
    "member_id": ["desynpuf_id", "bene", "beneficiary"],
    "claim_id": ["claim", "clm"],
    "date": ["date", "dt"],
    "payment": ["payment", "pmt", "paid", "reimb"],
    "diagnosis": ["diag", "icd"],
    "provider": ["provider", "prvdr"],
}

for label, terms in keywords.items():
    print(f"\nBENEFICIARY - {label}")
    print(search_columns(beneficiary_df, terms))

    print(f"INPATIENT - {label}")
    print(search_columns(inpatient_df, terms))

    print(f"OUTPATIENT - {label}")
    print(search_columns(outpatient_df, terms))

In [ ]:
# Helper function to safely select columns

def get_first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col

    return None

In [ ]:
# Map inpatient columns to clean names

inpatient_column_map = {
    "member_id": get_first_existing_column(
        inpatient_df,
        ["DESYNPUF_ID", "BENE_ID"],
    ),
    "claim_id": get_first_existing_column(
        inpatient_df,
        ["CLM_ID"],
    ),
    "claim_start_date": get_first_existing_column(
        inpatient_df,
        ["CLM_FROM_DT", "CLM_THRU_DT"],
    ),
    "claim_end_date": get_first_existing_column(
        inpatient_df,
        ["CLM_THRU_DT"],
    ),
    "claim_payment_amount": get_first_existing_column(
        inpatient_df,
        ["CLM_PMT_AMT", "CLM_TOT_CHRG_AMT", "NCH_PRMRY_PYR_CLM_PD_AMT"],
    ),
    "provider_id": get_first_existing_column(
        inpatient_df,
        ["PRVDR_NUM"],
    ),
    "primary_diagnosis_code": get_first_existing_column(
        inpatient_df,
        ["ADMTNG_ICD9_DGNS_CD", "ICD9_DGNS_CD_1"],
    ),
}

inpatient_column_map

In [ ]:
# Map outpatient columns to clean names

outpatient_column_map = {
    "member_id": get_first_existing_column(
        outpatient_df,
        ["DESYNPUF_ID", "BENE_ID"],
    ),
    "claim_id": get_first_existing_column(
        outpatient_df,
        ["CLM_ID"],
    ),
    "claim_start_date": get_first_existing_column(
        outpatient_df,
        ["CLM_FROM_DT", "CLM_THRU_DT"],
    ),
    "claim_end_date": get_first_existing_column(
        outpatient_df,
        ["CLM_THRU_DT"],
    ),
    "claim_payment_amount": get_first_existing_column(
        outpatient_df,
        ["CLM_PMT_AMT", "CLM_TOT_CHRG_AMT", "NCH_PRMRY_PYR_CLM_PD_AMT"],
    ),
    "provider_id": get_first_existing_column(
        outpatient_df,
        ["PRVDR_NUM"],
    ),
    "primary_diagnosis_code": get_first_existing_column(
        outpatient_df,
        ["ICD9_DGNS_CD_1"],
    ),
}

outpatient_column_map

In [ ]:
# Build clean inpatient claims dataset

clean_inpatient_df = pd.DataFrame()

for clean_col, source_col in inpatient_column_map.items():
    if source_col is None:
        clean_inpatient_df[clean_col] = np.nan
    else:
        clean_inpatient_df[clean_col] = inpatient_df[source_col]

clean_inpatient_df["claim_type"] = "inpatient"

print("Clean inpatient shape:", clean_inpatient_df.shape)
display(clean_inpatient_df.head())

In [ ]:
# Build clean outpatient claims dataset

clean_outpatient_df = pd.DataFrame()

for clean_col, source_col in outpatient_column_map.items():
    if source_col is None:
        clean_outpatient_df[clean_col] = np.nan
    else:
        clean_outpatient_df[clean_col] = outpatient_df[source_col]

clean_outpatient_df["claim_type"] = "outpatient"

print("Clean outpatient shape:", clean_outpatient_df.shape)
display(clean_outpatient_df.head())

In [ ]:
# Combine inpatient and outpatient claims

claims_df = pd.concat(
    [
        clean_inpatient_df,
        clean_outpatient_df,
    ],
    ignore_index=True,
)

print("Combined claims shape:", claims_df.shape)
display(claims_df.head())

In [ ]:
# Clean text fields

text_cols = [
    "member_id",
    "claim_id",
    "provider_id",
    "primary_diagnosis_code",
    "claim_type",
]

for col in text_cols:
    claims_df[col] = (
        claims_df[col]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
        })
    )

display(claims_df.head())

In [ ]:
# Clean date fields

date_cols = [
    "claim_start_date",
    "claim_end_date",
]

for col in date_cols:
    claims_df[col] = pd.to_datetime(
        claims_df[col],
        errors="coerce",
    )

claims_df["claim_duration_days"] = (
    claims_df["claim_end_date"] - claims_df["claim_start_date"]
).dt.days + 1

claims_df["claim_duration_days"] = claims_df["claim_duration_days"].clip(lower=1)

display(
    claims_df[
        [
            "claim_start_date",
            "claim_end_date",
            "claim_duration_days",
        ]
    ].head()
)

In [ ]:
# Clean payment amount

def clean_amount(value):
    if pd.isna(value):
        return np.nan

    value = str(value).replace("$", "").replace(",", "").strip()

    try:
        return float(value)
    except ValueError:
        return np.nan


claims_df["claim_payment_amount"] = claims_df["claim_payment_amount"].apply(clean_amount)

display(claims_df["claim_payment_amount"].describe())

In [ ]:
# Check missing values after cleaning

missing_summary = claims_df.isna().sum().reset_index()
missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(claims_df) * 100).round(2)

display(missing_summary.sort_values("missing_pct", ascending=False))

In [ ]:
# Remove rows missing core claim fields

before_rows = len(claims_df)

required_cols = [
    "member_id",
    "claim_id",
    "claim_payment_amount",
    "claim_type",
]

claims_df = claims_df.dropna(subset=required_cols).copy()

after_rows = len(claims_df)

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)

display(claims_df.head())

In [ ]:
# Basic claims summary

print("Total claims:", len(claims_df))
print("Unique members:", claims_df["member_id"].nunique())
print("Unique providers:", claims_df["provider_id"].nunique())
print("Claim types:", claims_df["claim_type"].value_counts().to_dict())

display(claims_df["claim_payment_amount"].describe())
display(claims_df["claim_type"].value_counts().to_frame("count"))

In [ ]:
# Create high-cost claim target
# This will be used in Notebook 3 for modeling.

high_cost_threshold = claims_df["claim_payment_amount"].quantile(0.90)

claims_df["high_cost_claim"] = (
    claims_df["claim_payment_amount"] >= high_cost_threshold
).astype(int)

print("High-cost threshold:", high_cost_threshold)
display(claims_df["high_cost_claim"].value_counts(normalize=True).to_frame("rate"))
display(claims_df[["claim_payment_amount", "high_cost_claim"]].head())

In [ ]:
# Create member-level utilization features

member_features_df = (
    claims_df
    .groupby("member_id")
    .agg(
        total_claims=("claim_id", "nunique"),
        total_claim_payment=("claim_payment_amount", "sum"),
        avg_claim_payment=("claim_payment_amount", "mean"),
        max_claim_payment=("claim_payment_amount", "max"),
        inpatient_claims=("claim_type", lambda x: (x == "inpatient").sum()),
        outpatient_claims=("claim_type", lambda x: (x == "outpatient").sum()),
        unique_providers=("provider_id", "nunique"),
        unique_diagnoses=("primary_diagnosis_code", "nunique"),
        avg_claim_duration_days=("claim_duration_days", "mean"),
        high_cost_claims=("high_cost_claim", "sum"),
    )
    .reset_index()
)

member_features_df["has_high_cost_claim"] = (
    member_features_df["high_cost_claims"] > 0
).astype(int)

display(member_features_df.head())

In [ ]:
# Create claim-level modeling features

claim_features_df = claims_df.copy()

claim_features_df["claim_payment_log"] = np.log1p(claim_features_df["claim_payment_amount"])
claim_features_df["has_provider_id"] = claim_features_df["provider_id"].notna().astype(int)
claim_features_df["has_diagnosis_code"] = claim_features_df["primary_diagnosis_code"].notna().astype(int)

display(claim_features_df.head())

In [ ]:
# Save cleaned claims and features

clean_claims_output_path = PREPROCESSED_DIR / "clean_claims_data.csv"
claim_features_output_path = FEATURES_DIR / "claim_features.csv"
member_features_output_path = FEATURES_DIR / "member_claim_features.csv"

claims_df.to_csv(clean_claims_output_path, index=False)
claim_features_df.to_csv(claim_features_output_path, index=False)
member_features_df.to_csv(member_features_output_path, index=False)

print("Saved:", clean_claims_output_path)
print("Saved:", claim_features_output_path)
print("Saved:", member_features_output_path)

In [ ]:
# Reload saved files to confirm they work

check_claims_df = pd.read_csv(clean_claims_output_path)
check_claim_features_df = pd.read_csv(claim_features_output_path)
check_member_features_df = pd.read_csv(member_features_output_path)

print("Clean claims shape:", check_claims_df.shape)
print("Claim features shape:", check_claim_features_df.shape)
print("Member features shape:", check_member_features_df.shape)

display(check_claim_features_df.head())

In [ ]:
# Final notebook summary

print("Notebook 2 complete.")
print("Clean claim records:", len(claims_df))
print("Unique members:", claims_df["member_id"].nunique())
print("High-cost threshold:", high_cost_threshold)

print("\nFiles created:")
print("1. data/02_preprocessed/clean_claims_data.csv")
print("2. data/03_features/claim_features.csv")
print("3. data/03_features/member_claim_features.csv")